In [11]:
# Step 1. Basic setup and paths

import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)

# Notebook is inside Cohort_Project/notebooks
PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data_raw"
DATA_PROCESSED = PROJECT_ROOT / "data_processed"
OUTPUTS = PROJECT_ROOT / "outputs"

print("Project root:", PROJECT_ROOT)
print("Raw files:", sorted([p.name for p in DATA_RAW.glob("*.xlsx")]))

Project root: /Users/krishnapatel/Documents/Cohort_Project
Raw files: ['Chronic_Disease_Hospitalization_Snapshot_Data_2014_2023.xlsx', 'Health_Behaviours_Alcohol_Snapshot_Data_2015-2020.xlsx', 'Health_Behaviours_Nutrition_Healthy_Weights_Snapshot_Data_2015-2021.xlsx', 'Health_Behaviours_PhysAct_Snapshot_Data_2015-2021.xlsx', 'Health_Behaviours_Smoking_Snapshot_Data_2015-2021.xlsx']


In [12]:
# Step 2. Load all datasets

hosp = pd.read_excel(
    DATA_RAW / "Chronic_Disease_Hospitalization_Snapshot_Data_2014_2023.xlsx",
    header=2,
    engine="openpyxl"
)

smoke = pd.read_excel(
    DATA_RAW / "Health_Behaviours_Smoking_Snapshot_Data_2015-2021.xlsx",
    header=2,
    engine="openpyxl"
)

pa = pd.read_excel(
    DATA_RAW / "Health_Behaviours_PhysAct_Snapshot_Data_2015-2021.xlsx",
    header=2,
    engine="openpyxl"
)

nut = pd.read_excel(
    DATA_RAW / "Health_Behaviours_Nutrition_Healthy_Weights_Snapshot_Data_2015-2021.xlsx",
    header=2,
    engine="openpyxl"
)

alcohol = pd.read_excel(
    DATA_RAW / "Health_Behaviours_Alcohol_Snapshot_Data_2015-2020.xlsx",
    header=2,
    engine="openpyxl"
)

print("Loaded shapes:")
print("hosp:", hosp.shape)
print("smoke:", smoke.shape)
print("pa:", pa.shape)
print("nut:", nut.shape)
print("alcohol:", alcohol.shape)

Loaded shapes:
hosp: (51800, 9)
smoke: (6216, 9)
pa: (4884, 9)
nut: (16872, 9)
alcohol: (16686, 9)


In [13]:
# Step 3. Define 4-year mapping

cycle_to_years = {
    "2015-16": [2016, 2017],
    "2017-18": [2018, 2019]
}

In [14]:
# Step 4. Create extraction function for risk factors

def extract_risk_panel(df, indicator, new_col):
    rows = []
    for cycle, years in cycle_to_years.items():
        temp = df[
            (df["Indicator"].astype(str).str.strip() == indicator) &
            (df["Cycle"].astype(str).str.strip() == cycle) &
            (df["Measure"].astype(str).str.strip() == "Age-standardized rate (both sexes)")
        ][["Geography", "Rate"]].copy()

        temp = temp[~temp["Geography"].str.contains("Ontario", case=False, na=False)]
        temp.columns = ["PHU", new_col]

        for year in years:
            t = temp.copy()
            t["Year"] = year
            rows.append(t)

    return pd.concat(rows, ignore_index=True)

In [15]:
# Step 5. Extract all four risk panels

smoke_panel = extract_risk_panel(
    smoke,
    "Self-reported adult daily smoking rate",
    "smoking_rate"
)

pa_panel = extract_risk_panel(
    pa,
    "Self-reported no adult physical activity minutes",
    "no_pa_minutes_rate"
)

nut_panel = extract_risk_panel(
    nut,
    "Self-reported prevalence of consumption of vegetables and fruits four or less times per day",
    "low_fv_rate"
)

alcohol_panel = extract_risk_panel(
    alcohol,
    "Self-reported rate of exceeding either low-risk alcohol drinking guideline",
    "heavy_drinking_rate"
)

print("Risk years check:")
print("smoke:", sorted(smoke_panel["Year"].unique()))
print("pa:", sorted(pa_panel["Year"].unique()))
print("nut:", sorted(nut_panel["Year"].unique()))
print("alcohol:", sorted(alcohol_panel["Year"].unique()))

Risk years check:
smoke: [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019)]
pa: [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019)]
nut: [np.int64(2016), np.int64(2017)]
alcohol: [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019)]


In [16]:
# Step 6. Merge risk panels first

risk_panel = smoke_panel.merge(pa_panel, on=["PHU", "Year"], how="inner")
risk_panel = risk_panel.merge(nut_panel, on=["PHU", "Year"], how="inner")
risk_panel = risk_panel.merge(alcohol_panel, on=["PHU", "Year"], how="inner")

print("Risk panel rows:", len(risk_panel))
print("Unique PHUs:", risk_panel["PHU"].nunique())
print("Years:", sorted(risk_panel["Year"].unique()))

Risk panel rows: 44
Unique PHUs: 22
Years: [np.int64(2016), np.int64(2017)]


In [17]:
# Step 7. Extract hospitalization for 2016 to 2019

hosp_panel = hosp[
    (hosp["Indicator"].astype(str).str.strip() == "Hospitalization for stroke") &
    (hosp["Measure"].astype(str).str.strip() == "Age-standardized rate (both sexes)") &
    (hosp["Year"].between(2016, 2019))
][["Geography", "Year", "Rate"]].copy()

hosp_panel = hosp_panel[~hosp_panel["Geography"].str.contains("Ontario", case=False, na=False)]
hosp_panel.columns = ["PHU", "Year", "stroke_hosp_rate"]

print("Hosp panel rows:", len(hosp_panel))
print("Hosp years:", sorted(hosp_panel["Year"].unique()))

Hosp panel rows: 140
Hosp years: [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019)]


In [18]:
# Step 8. Merge final panel dataset

panel_df = hosp_panel.merge(risk_panel, on=["PHU", "Year"], how="inner")

print("Final panel rows:", len(panel_df))
print("Unique PHUs:", panel_df["PHU"].nunique())
print("Years:", sorted(panel_df["Year"].unique()))

panel_df.head()

Final panel rows: 44
Unique PHUs: 22
Years: [np.int64(2016), np.int64(2017)]


,PHU,Year,stroke_hosp_rate,smoking_rate,no_pa_minutes_rate,low_fv_rate,heavy_drinking_rate
0,Algoma Public Health,2016,127.8,26.2,19.0,73.2,55.1
1,Algoma Public Health,2017,131.1,26.2,19.0,73.2,55.1
2,Durham Region Health Department,2016,103.8,14.6,19.7,71.6,51.0
3,Durham Region Health Department,2017,115.2,14.6,19.7,71.6,51.0
4,Halton Region Public Health,2016,86.7,11.2,22.4,72.2,49.5


In [19]:
out_path = DATA_PROCESSED / "panel_dataset_2016_2019.csv"
panel_df.to_csv(out_path, index=False)
print("Saved:", out_path)

Saved: /Users/krishnapatel/Documents/Cohort_Project/data_processed/panel_dataset_2016_2019.csv
